# Model 1 — clinical fine-tune (continues a saved run)

Takes the baseline saved by the training notebook and fine-tunes it on **89 wound
outlines drawn by hand across 31 wounds and two hospitals**, then judges the result
**in centimetres** the way the app measures, and exports only if it passes.

No GPU hours are spent rebuilding the baseline — that part is already done.

**Attach:** `diafootcare-model1` (the saved `.keras`), `diafootcare-wound-outlines`,
`fuseg-wound`, `dfutissue`.  **Accelerator:** GPU T4 ×2.

## 1 · Find everything, load the outlined wounds

In [ ]:
# ============================================================
# 1/5 — find everything, load the hand-outlined wounds
# ============================================================
# This notebook continues from a run whose outputs were saved: it needs no GPU
# hours to rebuild the baseline. Every path is searched for rather than trusted,
# because Kaggle mounts datasets under more than one shape and renames slugs.
import json, os, glob
import numpy as np, cv2, tensorflow as tf

IMG_H = IMG_W = 384          # the deployed input size; changing it changes the app


def find_dir(*candidates, tail=None, marker=None):
    for c in candidates:
        if c and os.path.isdir(c):
            return c
    for root, dirs, files in os.walk("/kaggle/input"):
        r = root.replace("\\", "/")
        if tail and r.endswith(tail):
            return root
        if marker and marker in files:
            return root
    return None


def find_file(name):
    for root, _d, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    return name if os.path.exists(name) else None


CLIN_DIR = find_dir("/kaggle/input/datasets/ibrahimshehada/diafootcare-wound-outlines",
                    "/kaggle/input/diafootcare-wound-outlines", marker="index.json")
assert CLIN_DIR, "attach diafootcare-wound-outlines"

FUSEG_TR_IMG = find_dir(tail="FUSeg/train/images")
FUSEG_TR_LBL = find_dir(tail="FUSeg/train/labels")
FUSEG_VA_IMG = find_dir(tail="FUSeg/validation/images")
FUSEG_VA_LBL = find_dir(tail="FUSeg/validation/labels")
DFU_IMG = find_dir(tail="DFUTissue/Labeled/Original/Images/TrainVal")
DFU_LBL = DFU_IMG.replace("Images", "Annotations") if DFU_IMG else None

# The baseline from the previous run. unet_model.keras is what that notebook chose
# as Model 1; unet_phase2.keras is the pre-self-training checkpoint.
BASE_KERAS = find_file("unet_model.keras") or find_file("unet_phase2.keras")

print("clinical    :", CLIN_DIR)
print("FUSeg train :", FUSEG_TR_IMG or "—")
print("FUSeg val   :", FUSEG_VA_IMG or "—   (the forgetting check will be SKIPPED)")
print("DFUTissue   :", DFU_IMG or "—")
print("base weights:", BASE_KERAS or "—")
assert BASE_KERAS, "upload unet_model.keras / unet_phase2.keras as a dataset first"

_index = json.load(open(os.path.join(CLIN_DIR, "index.json"), encoding="utf-8"))
_splits = json.load(open(os.path.join(CLIN_DIR, "splits.json"), encoding="utf-8"))
_gate_wounds = list(json.load(open(os.path.join(CLIN_DIR, "gate.json"), encoding="utf-8")))
clin_tr = [it for it in _index if it["wound"] in _splits["train"]]
clin_va = [it for it in _index if it["wound"] in _splits["val"]]


def _clin_pair(it):
    img = cv2.cvtColor(cv2.imread(os.path.join(CLIN_DIR, "images", it["file"])), cv2.COLOR_BGR2RGB)
    msk = cv2.imread(os.path.join(CLIN_DIR, "masks", it["file"]), cv2.IMREAD_GRAYSCALE)
    if img.shape[0] != IMG_H:
        img = cv2.resize(img, (IMG_W, IMG_H))
        msk = cv2.resize(msk, (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST)
    return img.astype(np.float32) / 255.0, (msk > 127).astype(np.float32)[..., None]


Xc_tr = np.array([_clin_pair(it)[0] for it in clin_tr], np.float32)
Yc_tr = np.array([_clin_pair(it)[1] for it in clin_tr], np.float32)
Xc_va = np.array([_clin_pair(it)[0] for it in clin_va], np.float32)
Yc_va = np.array([_clin_pair(it)[1] for it in clin_va], np.float32)

print(f"\nclinical: train {len(Xc_tr)} / val {len(Xc_va)} photographs "
      f"({len(_splits['train'])}/{len(_splits['val'])} wounds, split BY WOUND)")
print(f"gate: {len(_gate_wounds)} wounds the current model already measures well")


def load_dir(img_dir, lbl_dir, cap=None):
    """Masks may be 0/1, 0/255 or a label map — anything above zero is wound."""
    if not img_dir or not lbl_dir:
        return (np.empty((0, IMG_H, IMG_W, 3), np.float32),
                np.empty((0, IMG_H, IMG_W, 1), np.float32))
    X, Y = [], []
    for name in sorted(os.listdir(img_dir)):
        mp = os.path.join(lbl_dir, name)
        if not os.path.exists(mp):
            for e in (".png", ".jpg", ".jpeg"):
                if os.path.exists(os.path.splitext(mp)[0] + e):
                    mp = os.path.splitext(mp)[0] + e
                    break
            else:
                continue
        im = cv2.imread(os.path.join(img_dir, name))
        mk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if im is None or mk is None:
            continue
        X.append(cv2.resize(cv2.cvtColor(im, cv2.COLOR_BGR2RGB),
                            (IMG_W, IMG_H)).astype(np.float32) / 255.0)
        Y.append((cv2.resize(mk, (IMG_W, IMG_H),
                             interpolation=cv2.INTER_NEAREST) > 0).astype(np.float32)[..., None])
        if cap and len(X) >= cap:
            break
    return np.asarray(X, np.float32), np.asarray(Y, np.float32)


## 2 · Losses, the model, and the BEFORE measurement

In [ ]:
# ============================================================
# 2/5 — losses, the model, and the BEFORE measurement
# ============================================================
# Judged in CENTIMETRES, not Dice. A mask can gain Dice while losing the extent
# that gets measured, and centimetres are what the clinician is handed. Every step
# below runs in ai_service.dart, the printed-label guard included.
import os
import numpy as np, cv2, tensorflow as tf
from tensorflow.keras import backend as K

AUTOTUNE = tf.data.AUTOTUNE
BATCH = 8


def dice_coef(y_true, y_pred, smooth=1.0):
    yt, yp = K.flatten(K.cast(y_true, "float32")), K.flatten(K.cast(y_pred, "float32"))
    return (2.0 * K.sum(yt * yp) + smooth) / (K.sum(yt) + K.sum(yp) + smooth)


def iou_metric(y_true, y_pred, smooth=1.0):
    yt = K.flatten(K.cast(y_true, "float32"))
    yp = K.flatten(K.cast(K.round(y_pred), "float32"))
    i = K.sum(yt * yp)
    return (i + smooth) / (K.sum(yt) + K.sum(yp) - i + smooth)


def focal_tversky_loss(y_true, y_pred, alpha=0.3, beta=0.7, gamma=0.75, smooth=1.0):
    # beta > alpha punishes MISSED wound pixels harder. The wound is a median 0.9%
    # of the frame here, and every failure diagnosed in the clinic was the model
    # measuring too little.
    yt, yp = K.flatten(K.cast(y_true, "float32")), K.flatten(K.cast(y_pred, "float32"))
    tp, fn, fp = K.sum(yt * yp), K.sum(yt * (1 - yp)), K.sum((1 - yt) * yp)
    return K.pow(1.0 - (tp + smooth) / (tp + alpha * fp + beta * fn + smooth), gamma)


def seg_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(
        tf.cast(y_true, "float32"), tf.cast(y_pred, "float32")))
    return bce + focal_tversky_loss(y_true, y_pred)


model = tf.keras.models.load_model(BASE_KERAS, compile=False)
model.compile(optimizer="adam", loss=seg_loss, metrics=[dice_coef, iou_metric])
# The training notebook keeps a handle on the encoder for freeze/unfreeze; a model
# loaded from disk has none, so find the nested MobileNetV2 by type.
if not hasattr(model, "backbone"):
    nested = [l for l in model.layers if isinstance(l, tf.keras.Model)]
    if nested:
        model.backbone = nested[0]
        print("encoder handle:", model.backbone.name)
print("loaded:", BASE_KERAS, "| input", model.input_shape)

# FUSeg validation — the original domain. Without it, forgetting would go unnoticed.
Xv, Yv = load_dir(FUSEG_VA_IMG, FUSEG_VA_LBL)
val_ds = tf.data.Dataset.from_tensor_slices((Xv, Yv)).batch(BATCH).prefetch(AUTOTUNE) \
    if len(Xv) else None
fuseg_before = None
if val_ds is not None:
    fuseg_before = model.evaluate(val_ds, verbose=0, return_dict=True)["dice_coef"]
    print(f"FUSeg val: {len(Xv)} images, Dice {fuseg_before:.3f}")
else:
    print("no FUSeg validation found — the forgetting check will be SKIPPED and said so")


def _measure_cm(prob, item):
    m = (prob >= 0.5).astype(np.uint8)
    if m.sum() == 0:                                   # 0.5x-peak fallback
        pk = float(prob.max())
        if pk <= 0:
            return None
        m = (prob > 0.5 * pk).astype(np.uint8)
    k = np.ones((5, 5), np.uint8)
    m = cv2.morphologyEx(cv2.morphologyEx(m, cv2.MORPH_OPEN, k), cv2.MORPH_CLOSE, k)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return None

    bgr = cv2.imread(os.path.join(CLIN_DIR, "images", item["file"]))
    hsv = cv2.cvtColor(cv2.resize(bgr, (m.shape[1], m.shape[0])), cv2.COLOR_BGR2HSV)
    paper = (hsv[..., 1] < 50) & (hsv[..., 2] > 170)   # printed ink sits on a white card
    keep = []
    for i in range(1, n):
        if st[i, cv2.CC_STAT_AREA] < 10:
            continue
        comp = (lab == i).astype(np.uint8)
        collar = (cv2.dilate(comp, k, iterations=2) > 0) & (comp == 0)
        if collar.sum() and paper[collar].mean() >= 0.40:
            continue                                   # that blob is the label
        keep.append(i)
    if not keep:
        return None

    idx = max(keep, key=lambda i: st[i, cv2.CC_STAT_AREA])
    ys, xs = np.where(lab == idx)
    px, py = item.get("ppc_x"), item.get("ppc_y")      # pixels per cm, from the ring
    if not px or not py:
        return None
    sx, sy = m.shape[1] / IMG_W, m.shape[0] / IMG_H
    p = np.stack([xs * sx / px, ys * sy / py], 1).astype(np.float64)
    p -= p.mean(0)
    _, _, vt = np.linalg.svd(p, full_matrices=False)   # principal axis = longest extent
    q = p @ vt.T
    return float(q[:, 0].max() - q[:, 0].min())


def clinical_report(net, title):
    per, dices, unmeasured = {}, [], 0
    P = net.predict(Xc_va, batch_size=BATCH, verbose=0)[..., 0]
    Pf = net.predict(Xc_va[:, :, ::-1, :], batch_size=BATCH, verbose=0)[..., 0][:, :, ::-1]
    P = (P + Pf) / 2.0                                 # 2-view TTA, as the app does
    for i, it in enumerate(clin_va):
        gt, pr = Yc_va[i, ..., 0] > 0.5, P[i] >= 0.5
        dices.append(2 * (gt & pr).sum() / max(1e-6, gt.sum() + pr.sum()))
        if it.get("ref_quality") != "measured" or not it.get("ref_major"):
            continue                                   # an estimate by eye is not truth
        cm = _measure_cm(P[i], it)
        if cm is None:
            unmeasured += 1
            continue
        per.setdefault(it["wound"], []).append(100.0 * (cm - it["ref_major"]) / it["ref_major"])
    w = {k: float(np.mean(np.abs(v))) for k, v in per.items()}
    print("\n--- " + title + " ---")
    print(f"  clinical Dice {np.mean(dices):.3f} | wounds scored: {len(w)} | "
          f"unmeasurable: {unmeasured}")
    for k, v in sorted(w.items(), key=lambda kv: -kv[1]):
        print(f"    {k:<42}{v:6.1f}%")
    if w:
        print(f"  MEAN ABSOLUTE ERROR: {np.mean(list(w.values())):.1f}%")
    return w, float(np.mean(dices))


before, cdice_before = clinical_report(model, "BEFORE clinical fine-tune")


## 3 · Fine-tune

In [ ]:
# ============================================================
# 3/5 — fine-tune on the outlined wounds
# ============================================================
# 89 clinical photographs against ~1,100 the model already learned from. Ours alone
# would score well on our own split and collapse everywhere else, so each epoch
# mixes them with the same precise pool the training notebook used.
import os
import numpy as np, tensorflow as tf
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam

CLIN_OVERSAMPLE = 4
FINE_TUNE_FROM = 40          # same unfreeze point as the training notebook
MAX_FUSEG = None             # set an int (e.g. 600) if the session runs out of RAM


def augment(x, y):
    """Geometry and light only. No elastic warping: here the boundary IS the label."""
    if tf.random.uniform([]) < 0.5:
        x, y = tf.image.flip_left_right(x), tf.image.flip_left_right(y)
    if tf.random.uniform([]) < 0.5:
        x, y = tf.image.flip_up_down(x), tf.image.flip_up_down(y)
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    x, y = tf.image.rot90(x, k), tf.image.rot90(y, k)
    x = tf.image.random_brightness(x, 0.12)
    x = tf.image.random_contrast(x, 0.9, 1.1)
    return tf.clip_by_value(x, 0.0, 1.0), y


def make_ds(X, Y, training, batch=BATCH):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    if training:
        ds = ds.shuffle(min(len(X), 512), seed=42).map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(batch).prefetch(AUTOTUNE)


def make_opt(lr):
    o = Adam(lr)
    pol = tf.keras.mixed_precision.global_policy().name
    return tf.keras.mixed_precision.LossScaleOptimizer(o) if "float16" in pol else o


def make_callbacks(path, patience, lr_patience=5):
    return [
        tf.keras.callbacks.ModelCheckpoint(path, monitor="val_dice_coef", mode="max",
                                           save_best_only=True, verbose=0),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_dice_coef", mode="max", factor=0.5,
                                             patience=lr_patience, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_dice_coef", mode="max", patience=patience,
                                         min_delta=1e-4, restore_best_weights=True, verbose=1),
    ]


Xf, Yf = load_dir(FUSEG_TR_IMG, FUSEG_TR_LBL, MAX_FUSEG)
Xd, Yd = load_dir(DFU_IMG, DFU_LBL)
print(f"precise pool: FUSeg {len(Xf)} + DFUTissue {len(Xd)}")
pools_X = [a for a in (Xf, Xd) if len(a)]
pools_Y = [a for a in (Yf, Yd) if len(a)]
if not pools_X:
    print("!! no precise pool — training on 89 photographs ALONE.")
    print("   The model may forget what it knows; the gate is the only thing that")
    print("   will tell you, so do not skip it.")

X_mix = np.concatenate([np.repeat(Xc_tr, CLIN_OVERSAMPLE, 0)] + pools_X, 0)
Y_mix = np.concatenate([np.repeat(Yc_tr, CLIN_OVERSAMPLE, 0)] + pools_Y, 0)
print(f"mixed pool: {len(Xc_tr)}x{CLIN_OVERSAMPLE} clinical + "
      f"{sum(len(a) for a in pools_X)} precise = {len(X_mix)}")

tr_ds = make_ds(X_mix, Y_mix, training=True)
va_ds = make_ds(Xc_va, Yc_va, training=False)

# Stage 1 — decoder only. A general encoder should not be moved by 89 photographs
# while the decoder is still adjusting to a new boundary convention.
if hasattr(model, "backbone"):
    model.backbone.trainable = False
model.compile(optimizer=make_opt(3e-4), loss=seg_loss,
              metrics=["accuracy", dice_coef, iou_metric])
print("\n[Stage 1] decoder only, encoder frozen")
model.fit(tr_ds, validation_data=va_ds, epochs=12,
          callbacks=make_callbacks("unet_clinical.keras", patience=6))

# Stage 2 — deep unfreeze at a low LR: adjust the boundary, do not overwrite the
# model. Encoder BatchNorm stays frozen so ImageNet running statistics hold.
if hasattr(model, "backbone"):
    model.backbone.trainable = True
    for i, layer in enumerate(model.backbone.layers):
        if i < FINE_TUNE_FROM or isinstance(layer, BatchNormalization):
            layer.trainable = False
model.compile(optimizer=make_opt(2e-5), loss=seg_loss,
              metrics=["accuracy", dice_coef, iou_metric])
print("\n[Stage 2] deep fine-tune, low LR")
model.fit(tr_ds, validation_data=va_ds, epochs=25,
          callbacks=make_callbacks("unet_clinical.keras", patience=10))

if os.path.exists("unet_clinical.keras"):
    model = tf.keras.models.load_model("unet_clinical.keras", compile=False)
    model.compile(optimizer="adam", loss=seg_loss, metrics=[dice_coef, iou_metric])
    if not hasattr(model, "backbone"):
        nested = [l for l in model.layers if isinstance(l, tf.keras.Model)]
        if nested:
            model.backbone = nested[0]
    print("\nloaded the best clinical checkpoint")


## 4 · The gate

In [ ]:
# ============================================================
# CLINICAL 4/4 — the gate, then save only if it passes
# ============================================================
import json
import numpy as np, tensorflow as tf

after, cdice_after = clinical_report(model, "AFTER clinical fine-tune")
fuseg_after = (model.evaluate(val_ds, verbose=0, return_dict=True)["dice_coef"]
               if "val_ds" in globals() else None)

print("\n" + "=" * 74 + "\nGATE\n" + "=" * 74)
mb = np.mean(list(before.values())) if before else float("nan")
ma = np.mean(list(after.values())) if after else float("nan")
print(f"  clinical cm error   {mb:6.1f}%  ->  {ma:6.1f}%")
print(f"  clinical Dice       {cdice_before:6.3f}  ->  {cdice_after:6.3f}")
if fuseg_before is not None and fuseg_after is not None:
    print(f"  FUSeg val Dice      {fuseg_before:6.3f}  ->  {fuseg_after:6.3f}")
else:
    print("  FUSeg val Dice      NOT CHECKED — no val_ds in this session.")
    print("                      Forgetting the original domain would go unnoticed here.")

broke = []
print("\n  wounds the model already measured well:")
for w in _gate_wounds:
    # gate.json chooses WHICH wounds are checked. The baseline is this run's own
    # "before" pass, never a number from another script measured on other inputs.
    b, a = before.get(w), after.get(w)
    if b is None or a is None:
        print(f"    {w:<42} not scored this run")
        continue
    limit = max(b * 1.5, b + 5.0)          # tolerance, not a demand for perfection
    ok = a <= limit
    print(f"    {w:<42}{b:6.1f}% -> {a:6.1f}%   {'ok' if ok else 'REGRESSED'}")
    if not ok:
        broke.append(w)

forgot = (fuseg_before is not None and fuseg_after is not None
          and fuseg_after < fuseg_before - 0.03)
passed = (ma < mb) and not broke and not forgot

print("\n  " + ("PASS - unet_model.keras updated, the export cell will ship this"
                if passed else "FAIL - unet_model.keras left untouched"))
if not (ma < mb):
    print("    the clinical error did not improve")
for w in broke:
    print("    regressed: " + w)
if forgot:
    print(f"    forgot the original domain: FUSeg Dice {fuseg_before:.3f} -> {fuseg_after:.3f}")
if passed and fuseg_after is None:
    print("    NOTE: passed WITHOUT the forgetting check. Confirm on the original")
    print("          validation set before shipping this to anyone.")

json.dump(dict(before=before, after=after, clinical_dice=[cdice_before, cdice_after],
               fuseg_dice=[fuseg_before, fuseg_after], regressed=broke,
               forgot=bool(forgot), passed=bool(passed),
               forgetting_checked=fuseg_after is not None),
          open("gate_report.json", "w"), indent=1)
print("\n  wrote gate_report.json")

if passed:
    model.save("unet_model.keras")
    print("  saved -> unet_model.keras")
else:
    # Reload the pre-fine-tune model so the export cell ships the OLD weights.
    # A failed experiment must leave no trace in what reaches a patient.
    import os
    if os.path.exists("unet_model.keras"):
        model = tf.keras.models.load_model("unet_model.keras", compile=False)
        model.compile(optimizer="adam", loss=seg_loss, metrics=[dice_coef, iou_metric])
        print("  reloaded the pre-fine-tune model; nothing downstream changes.")


## 5 · Export

In [ ]:
# ============================================================
# 5/5 — export to TFLite, but only what passed
# ============================================================
# The model trains under a mixed_float16 policy, which leaves float16 ops in the
# graph that TFLite cannot legalize (ERROR_NEEDS_FLEX_OPS). The training notebook
# solves this by rebuilding the architecture in float32 and copying the weights
# across; here there is no build function, so the same thing is done from the
# model's own config with every dtype policy forced back to float32.
import json, pathlib, shutil, os
import numpy as np, tensorflow as tf

if not passed:
    print("The gate failed — nothing to export.")
    print("A model that fails the gate must not reach a patient, and the easiest way")
    print("for that to happen is an exported file sitting next to a passing one.")
else:
    def force_float32(node):
        if isinstance(node, dict):
            d = node.get("dtype")
            if isinstance(d, dict) or (isinstance(d, str) and "float16" in d):
                node["dtype"] = "float32"
            for v in node.values():
                force_float32(v)
        elif isinstance(node, list):
            for v in node:
                force_float32(v)

    tf.keras.mixed_precision.set_global_policy("float32")
    cfg = json.loads(model.to_json())
    force_float32(cfg)
    export_model = tf.keras.models.model_from_json(json.dumps(cfg))
    export_model.set_weights(model.get_weights())
    print("rebuilt a float32 copy:", export_model.output_shape)

    # prove the copy is the same network before it becomes the shipped file
    a = model.predict(Xc_va[:4], verbose=0)[..., 0]
    b = export_model.predict(Xc_va[:4], verbose=0)[..., 0]
    print(f"float32 copy vs trained model: max abs diff {np.abs(a - b).max():.6f}")

    SM = "sm_model1"
    if os.path.isdir(SM):
        shutil.rmtree(SM)
    export_model.export(SM)

    conv = tf.lite.TFLiteConverter.from_saved_model(SM)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.target_spec.supported_types = [tf.float16]
    pathlib.Path("model1_wound_fp16.tflite").write_bytes(conv.convert())
    size = os.path.getsize("model1_wound_fp16.tflite") / 1e6
    print(f"\nexported -> model1_wound_fp16.tflite   ({size:.1f} MB)")

    # and prove the exported file agrees with the model it came from
    itp = tf.lite.Interpreter(model_path="model1_wound_fp16.tflite")
    itp.allocate_tensors()
    i_, o_ = itp.get_input_details()[0], itp.get_output_details()[0]
    diffs = []
    for i in range(min(4, len(Xc_va))):
        itp.set_tensor(i_["index"], Xc_va[i:i + 1].astype(np.float32))
        itp.invoke()
        diffs.append(np.abs(itp.get_tensor(o_["index"])[0, ..., 0] - b[i]).mean())
    print(f"TFLite vs Keras: mean abs diff {np.mean(diffs):.5f}  "
          f"{'(fp16 rounding, as expected)' if np.mean(diffs) < 0.01 else '(!! too large)'}")
    print("\nDownload from the Output tab:")
    print("  model1_wound_fp16.tflite   -> assets/models/ in the app")
    print("  gate_report.json           -> the evidence")
    print("  unet_clinical.keras        -> so the next fine-tune can start here")
